In [51]:
# Подключаем библиотеки

import re
from dataclasses import dataclass
from collections import Counter

# Смотрим как делить, как строка или слово

TOKEN_RE = re.compile(r'"[^"]*"|[^\s"]+')

In [52]:
from IPython.display import display, HTML

# Действия
ACTIONS  = {"выбрать", "показать", "найти", "удалить"}

# Число

QUANT = {"все", "один"}

# Объект, как цель

OBJECTS  = {"машины", "автомобили", "авто", "автомобиль", "машину"}

# Характеристики и поля базы данных

FIELDS = {"цвет", "марка", "модель", "год", "пробег", "цена", "кузов",
          "коробка", "привод", "мощность", "владелец", "номер двигателя"}

# Операторы сравнения

OPS = {"равен", "равна", "равно", "больше", "меньше"}

# Ключевые слова связки

KEYWORDS = {"и", "или"}

# Род полей для согласования оператора "равен/равна/равно"
FIELD_GENDER = {
    "цвет": "m", "год": "m", "пробег": "m", "кузов": "m",
    "привод": "m", "владелец": "m", "номер двигателя": "m",
    "марка": "f", "модель": "f", "цена": "f",
    "коробка": "f", "мощность": "f"
}

# Правильные формы слова "равен" в зависимости от рода
GENDER_FORMS = {
    "m": "равен",
    "f": "равна",
    "n": "равно"
}

# Класс создания дерева

class Tree:
    def __init__(self, name, value="", children = None):
        self.name = name
        self.value = value
        self.children = children if children is not None else []

    # Добавляет дочерний узел

    def add(self, child):
        if child is not None:
            self.children.append(child)
        return self

    # Выводим дерево

    def print_tree(self, open_levels=2):
        def to_html(node, level=0):
            # Подсветка имени и значения
            if node.value:
                label = f'<span class="node-name">{node.name}</span>: <span class="node-value">«{node.value}»</span>'
            else:
                label = f'<span class="node-name">{node.name}</span>'

            # Если нет детей - просто лист
            if not node.children:
                return f'<span class="leaf-text">{label}</span>'

            # Ветка с возможностью сворачивания
            opened = "open" if level < open_levels else ""

            html = f'<details {opened}><summary>{label}</summary><ul>'
            for child in node.children:
                html += f'<li>{to_html(child, level + 1)}</li>'
            html += '</ul></details>'
            return html

        # Собираем итоговый HTML с элегантным CSS
        html = f"""
        <style>
            .tree-container {{
                font-family: 'Consolas', 'Courier New', monospace;
                font-size: 14px;
                color: #d4d4d4;
                background-color: #1e1e1e;
                padding: 20px;
                border-radius: 8px;
                line-height: 1.6;
                display: inline-block;
                min-width: 350px;
                box-shadow: 0 4px 6px rgba(0,0,0,0.3);
            }}
            .tree-container ul {{
                list-style: none;
                padding-left: 20px;
                margin: 0;
                position: relative;
            }}
            /* Скрываем отступ и линии для самого корня */
            .tree-container > ul {{
                padding-left: 0;
            }}
            .tree-container > ul > li {{
                padding-left: 0;
            }}
            .tree-container > ul > li::before,
            .tree-container > ul > li::after {{
                display: none !important;
            }}

            .tree-container li {{
                position: relative;
                padding-left: 20px;
            }}
            /* Вертикальная линия */
            .tree-container li::before {{
                content: "";
                position: absolute;
                top: 0;
                bottom: 0;
                left: 0;
                border-left: 1px solid #555;
            }}
            /* Обрезаем вертикальную линию для последнего ребенка (формируем уголок) */
            .tree-container li:last-child::before {{
                bottom: auto;
                height: 13px; /* Высота до горизонтальной линии */
            }}
            /* Горизонтальная линия */
            .tree-container li::after {{
                content: "";
                position: absolute;
                top: 13px; /* Выравнивание по центру строки */
                left: 0;
                width: 15px;
                border-top: 1px solid #555;
            }}

            /* Стили для summary (сворачиваемых узлов) */
            .tree-container details > summary {{
                cursor: pointer;
                list-style: none;
                display: inline-block;
                font-weight: 500;
            }}
            .tree-container details > summary::-webkit-details-marker {{
                display: none;
            }}
            /* Иконка-стрелочка */
            .tree-container details > summary::before {{
                content: "▶";
                position: absolute;
                left: 2px; /* Сдвигаем вправо от вертикальной линии */
                top: 3px;
                font-size: 10px;
                color: #888;
                transition: transform 0.2s;
            }}
            .tree-container details[open] > summary::before {{
                transform: rotate(90deg);
                top: 4px;
            }}

            /* Подсветка синтаксиса */
            .node-name {{
                color: #569cd6; /* Синий для ключей */
                font-weight: bold;
            }}
            .node-value {{
                color: #ce9178; /* Оранжевый для строк */
            }}
            .leaf-text {{
                color: #cccccc;
            }}
        </style>

        <div class="tree-container">
            <ul>
                <li>{to_html(self, 0)}</li>
            </ul>
        </div>
        """
        from IPython.display import display, HTML
        display(HTML(html))

In [53]:
# Основной класс разбора, разбирает предложение по уровням, ловит ошибки и анализирует их

# Класс ловит ошибку и выводит позицию
class ParseError(Exception):
    def __init__(self, msg, tok):
        super().__init__(f"{msg} (позиция {tok.pos})")
        self.tok = tok

# Основной класс обработки

class Parser:
    def __init__(self, tokens):

        # Фиксируем токен, тут идентификатор текущего токена и сам массив токенов

        self.i = 0
        self.tokens = tokens

    # Выдает текущий токен

    def peek(self):
        return self.tokens[self.i]

    # Создаем корень дерева

    def determine_root(self):
        root = Tree("Query")
        return root

    # Ожидает токен из списка допустимых значений. Если токен не соответствует - выбрасывает ParseError

    def expect(self, vals, what):

        # Получаем на вход допустимые значения, предупреждение, что может быть неправильно

        tok = self.peek()

        # Получаем токен

        self.validate_token(tok, vals, what)

        self.i += 1

        return tok

    # Обработка отсутствия токенов

    def pass_qua(self, *vals):
        cur = self.peek()

        # Фиксируем токен: если этого токена нет, но он подразумевается, то счётчик не сдвигается,
        # а возвращается None (значение по умолчанию подставляется в вызывающем коде).
        # Когда переходим к другому токену, по теории мы переходим к новому токену,
        # а в строке — на следующее слово; если мы просто неправильно ввели токен, то отработает ошибка.

        val = cur.value.lower() if cur.type != "STRING" else cur.value

        if val in vals:
            self.i += 1
            return cur
        return None

    # Обработка токена

    def validate_token(self, token, allowed_values, what="команда"):

        # Переводим в нижний регистр, проверяем, чтобы токен не был строкой

        val = token.value.lower() if token.type != "STRING" else token.value

        # Проверка есть ли токен в списке допустимых

        if val in allowed_values:
            return token

        # Обрабатываем конец запроса (EOF)

        found = token.value if token.type != "EOF" else "конец запроса"

        # Выводим допустимые значени и выводим ошибку

        expected = ", ".join(allowed_values)

        error_msg = (
            f" Неверная {what}.\n "
            f"Ожидалось: [{expected}].\n "
            f"Найдено: '{found}' (тип: {token.type})."
        )

        # Ловим исключение

        raise ParseError(error_msg, token)

    # Создаем ветку для условий

    def create_branch(self):

        # Фиксируем термы и операции; термы - это конкретное условие, которое включает
        # частицу "не", если она есть, характеристику, операцию сравнения, если она есть,
        # в противном случае ставим "равен", "равна" или "равно" в необходимом роде,
        # и описание характеристики

        terms = []
        operators = []

        # Первый обязательный терм
        terms.append(self.term())

        # Читаем пары: связка + терм
        while True:
            if self.peek().type == "EOF":
                break
            if self.peek().value.lower() not in KEYWORDS:
                break

            # Фиксируем операции и термы

            op = self.expect(KEYWORDS, "логическая связка (и/или)")
            operators.append(op.value.lower())
            terms.append(self.term())

        # Если только один терм
        if not operators:
            node = Tree("Condition")
            node.children.append(terms[0])
            return node

        # Сначала группируем по "и", потом соединяем каждую группу связкой или
        and_groups = []
        # Ставим первый терм
        current_group = [terms[0]]

        # Создаем группу отдельно, каждая группа содержит термы, смотрим термы соединенные связкой И, если эта группа закончилась, то закрываем и фиксируем группу и переходим к следующей

        for i, op in enumerate(operators):
            if op == "и":
                current_group.append(terms[i+1])
            else:  # "или"
                and_groups.append(current_group)
                current_group = [terms[i + 1]]

        # Если группа имеет размер 1, то тогда не нужно ее соединять связкой И

        and_groups.append(current_group)

        # Собираем финальное дерево
        def make_and_node(group):
            if len(group) == 1:
                return group[0] # Выделяем одиночный терм
            node = Tree("И")
            for t in group:
                node.children.append(t)
            return node

        # Теперь соединяем группы через "ИЛИ"
        if len(and_groups) == 1:
            result = make_and_node(and_groups[0])
        else:
            result = Tree("ИЛИ")
            for group in and_groups:
                result.children.append(make_and_node(group))

        condition = Tree("Condition")
        condition.children.append(result)
        return condition

    # Обработка одного терма

    def term(self):
        # Поле характеристика
        field_tok = self.expect(FIELDS, "поле (цвет/марка/год/...)")
        field_name = field_tok.value.lower()

       # Фиксируем род поля, ищем это поле в словаре, выводим род, потом сопоставляем род и оператор при выводе в дерево, чтобы род был согласован, если поле не найдено, то ставим мужской род

        gender = FIELD_GENDER.get(field_name, "m")
        correct_eq = GENDER_FORMS[gender]

        # Частица "не"
        has_not = self.pass_qua("не") is not None

        # Оператор (может отсутствовать)
        op_tok = self.pass_qua(*OPS)
        if op_tok:
            op_val = op_tok.value.lower()
            if op_val in {"равен", "равна", "равно"} and op_val != correct_eq:
                raise ParseError(
                    f"Согласование рода: для поля «{field_name}» нужно «{correct_eq}», а не «{op_val}»",
                    op_tok
                )
        else:
            # Если оператора нет - ставим правильную форму "равен/равна"
            op_val = correct_eq

        # Значение
        val_tok = self.peek()
        if val_tok.type == "EOF":
            raise ParseError("Ожидалось значение после оператора", val_tok)

        # Проверка чтобы токен типа WORD не был логической связкой
        if val_tok.type == "WORD" and val_tok.value.lower() in KEYWORDS:
            raise ParseError("Значение не может быть логической связкой (используйте кавычки)", val_tok)

        self.i += 1

        # Строим узел
        term_node = Tree("Term")
        term_node.add(Tree("Field", field_tok.value))
        term_node.add(Tree("Operator", ("не " + op_val) if has_not else op_val))
        term_node.add(Tree("Value", val_tok.value))

        return term_node

    # Метод производит парсинг, описывает порядок, содержит само описание структуры формального языка

    def parse(self):

        # Создаем корневой узел дерева

        root = self.determine_root()

        # Выводим действие в дерево

        act = self.expect(ACTIONS, "действие (выбрать/показать/найти/удалить)")

        root.children.append(Tree("Action", act.value))

        # Количество выводимых данных, базовое значение все
        q = self.pass_qua(*QUANT)

        # Обработка пропуска этой части

        if q:
            root.children.append(Tree("Quantifier", q.value))
        else:
            root.children.append(Tree("Quantifier", "все"))

        # Добавление объекта для поиска

        obj = self.expect(OBJECTS, "объект (машины/автомобили/авто/...)")
        root.children.append(Tree("Object", obj.value))

        # Определяем ресурс

        self.expect({"из"}, "слово «из»")
        self.expect({"таблицы"}, "слово «таблицы»")

        # Имя таблицы может быть и WORD, и NUMBER
        table_tok = self.peek()
        if table_tok.type == "EOF":
            raise ParseError("Ожидалось имя таблицы", table_tok)
        self.i += 1
        root.children.append(Tree("Source", table_tok.value))

        # Создаем ветку условия, проверка, что есть

        self.expect({"если"}, "слово «если»")

        condition = Tree("ConditionPart")

        # Создаем обработку ветвей с условием поиска

        condition.children.append(self.create_branch())
        root.children.append(condition)

        # Проверяем, что запрос полностью разобран (остался только EOF)

        if self.peek().type != "EOF":
            raise ParseError("Лишние символы в конце запроса", self.peek())

        return root


In [54]:
# Создаем класс токена, хранит тип, значение и позицию, для определения, где произошла ошибка

@dataclass
class Token:
    type: str
    value: str
    pos: int

In [55]:
# Разделяем на токены, токеном может выступать число, строка или слово отдельно
# Внедрена обработка кавычек, чтобы слово в кавычках считалось одним токеном, если не хотим получить ошибку в дальнейшем предлагаю учитывать в программе строгий порядок слов, чтобы пользователь вводил кавычки если название автомобиля сложное из 2-3 слов.
def tokenize(text):

    # Фиксируем токены

    tokens = []

    for m in TOKEN_RE.finditer(text):
        v = m.group()

        # Стоит заметить, что у нас 3 вида токенов STRING, NUMBER, WORD

        if v.startswith('"'):
            tokens.append(Token("STRING", v.strip('"'), m.start()))
        else:
            v = v.strip(".,!?;:")
            if v:
                try:
                    float(v)
                    t_type = "NUMBER"
                    # Обнаружил, что программа вещественные числа считает как WORD,
                    # а не NUMBER

                except ValueError:
                    t_type = "WORD"
                tokens.append(Token(t_type, v, m.start()))

    # Позволяет сказать, что это конец набора токенов для предложения

    tokens.append(Token("EOF", "", len(text)))

    return tokens

In [56]:
# Главная функция

def analyze(query: str):

    # Выводим запрос

    print("=" * 70)
    print("Запрос:", query)
    print("-" * 70)

   # Разбираем строку на токены, каждый токен это одно слово

    tokens = tokenize(query)
    if len(tokens) == 1:
        print("Результат: ОШИБКА")
        print("Пустой запрос")
        return

    # Производим парсинг строим деревья

    pars = Parser(tokens)

    # Выводим дерево

    try:
        pars.parse().print_tree()
    except ParseError as e:
        print("Результат: ОШИБКА ПАРСИНГА")
        print(e)
    except Exception as e:
        print("Результат: НЕИЗВЕСТНАЯ ОШИБКА")
        print(e)

In [57]:
# Список файлов с запросами
FILES = [
    "1_correct_easy.txt",
    "2_correct_medium.txt",
    "3_correct_hard.txt",
    "4_errors_easy.txt",
    "5_errors_medium.txt",
    "6_errors_hard.txt",
]
SHOW_TOKEN_TABLE = True


In [58]:
# Читаем файл

def load_queries(path):

    items, section = [], "Без раздела"
    with open(path, encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            if line.startswith("#"):
                if line.startswith("# ==="):
                    section = line.strip("# =").strip()
                continue
            items.append((line_no, section, line))
    return items

# Разбирает запрос без вывода деревьев

def check_query(query):
    tokens = tokenize(query)
    parser = Parser(tokens)

    try:
        result = parser.parse()
        result.print_tree()

        return tokens, True, None
    except Exception as e:
        return tokens, False, e

# Краткое описание ошибки: причина + токен и позиция

def error_text(e):
    reason = str(e).split("\n")[0].strip()
    tok = getattr(e, "tok", None)
    if tok is None:
        return reason
    found = tok.value if tok.type != "EOF" else "конец запроса"
    return f"{reason} | токен '{found}' ({tok.type}), позиция {tok.pos}"

In [59]:
# Собираем ститистику по файлу с примерами

def run_file(path):
    queries = load_queries(path)
    results = [] # номер, раздел, запрос, tokens, ok, err

    # Прогон запросов, уведомления
    print("=" * 90)
    print(f"Файл: {path}")
    print("=" * 90)
    for n, (line_no, section, query) in enumerate(queries, 1):
        tokens, ok, err = check_query(query)
        results.append((n, section, query, tokens, ok, err))
        if ok:
            print(f"[ OK    ] #{n:<3} {query}")
        else:
            print(f"[ ОШИБКА] #{n:<3} {query}")
            print(f"           -> {error_text(err)}")

    # 2. Статистика по файлу
    real_counts = []      # число токенов без EOF
    type_counter = Counter()
    for _, _, _, tokens, _, _ in results:
        real = [t for t in tokens if t.type != "EOF"]
        real_counts.append(len(real))
        type_counter.update(t.type for t in real)

    total = len(results)
    ok_count = sum(1 for r in results if r[4])
    total_tokens = sum(real_counts)

    print("\n" + "=" * 90)
    print("СТАТИСТИКА ПО ФАЙЛУ")
    print("=" * 90)
    print(f"Число предложений (запросов): {total}")
    print(f"успешно разобрано: {ok_count}")
    print(f"ошибок парсинга:  {total - ok_count}")
    print(f"Число токенов (без EOF): {total_tokens}")
    if total:
        print(f"Среднее токенов на запрос: {total_tokens / total:.1f}")
        print(f"Максимум / минимум: {max(real_counts)} / {min(real_counts)}")
    print("Токены по типам: " +
", ".join(f"{k}={type_counter.get(k, 0)}" for k in ("WORD", "NUMBER", "STRING")))

    # по разделам
    print("\nПо разделам:")
    sections = {}
    for _, section, _, _, ok, _ in results:
        s = sections.setdefault(section, [0, 0])
        s[0 if ok else 1] += 1
    w = max((len(s) for s in sections), default=10)
    for section, (o, e) in sections.items():
        print(f"  {section.ljust(w)}  OK: {o:<3} ОШИБКА: {e}")

    # 3. Сводная таблица по запросам (выводится всегда)
    print("\n" + "=" * 90)
    print("СВОДНАЯ ТАБЛИЦА ПО ЗАПРОСАМ")
    print("=" * 90)
    header = f"{'№':<4}{'Статус':<9}{'Токенов':<9}{'WORD':<6}{'NUMBER':<8}{'STRING':<8}Ошибка на токене"
    print(header)
    print("-" * len(header))
    for n, _, _, tokens, ok, err in results:
        real = [t for t in tokens if t.type != "EOF"]
        c = Counter(t.type for t in real)
        if ok:
            where = "-"
        else:
            tok = getattr(err, "tok", None)
            if tok is None:
                where = "не определён"
            else:
                found = tok.value if tok.type != "EOF" else "<конец запроса>"
                where = f"'{found}' поз. {tok.pos}"
        print(f"{n:<4}{'OK' if ok else 'ОШИБКА':<9}{len(real):<9}"
              f"{c.get('WORD', 0):<6}{c.get('NUMBER', 0):<8}{c.get('STRING', 0):<8}{where}")

    # 4. Подробная таблица токенов
    if SHOW_TOKEN_TABLE:
        print("\n" + "=" * 90)
        print("ТАБЛИЦА ТОКЕНОВ")
        print("=" * 90)
        print(f"{'Запрос':<8}{'Поз.':<7}{'Тип':<9}Значение")
        print("-" * 50)
        for n, _, _, tokens, _, _ in results:
            for t in tokens:
                val = t.value if t.type != "EOF" else "<EOF>"
                print(f"#{n:<7}{t.pos:<7}{t.type:<9}{val}")
            print("-" * 50)

    return results

In [60]:
# Обрабатываем файлы

def run_all():

    summary = []

    for path in FILES:
        try:
            res = run_file(path)
        except FileNotFoundError:
            print(f"\n!!! Файл не найден: {path}\n")
            continue
        tokens_total = sum(len([t for t in r[3] if t.type != "EOF"]) for r in res)
        ok = sum(1 for r in res if r[4])
        summary.append((path, len(res), ok, len(res) - ok, tokens_total))
        print("\n\n")

    print("=" * 90)
    print("ИТОГО ПО ВСЕМ ФАЙЛАМ")
    print("=" * 90)
    w = max((len(r[0]) for r in summary), default=10)
    print(f"{'Файл'.ljust(w)}  {'Запросов':<10}{'OK':<6}{'Ошибок':<8}Токенов")
    print("-" * (w + 36))
    for path, n, ok, err, tk in summary:
        print(f"{path.ljust(w)}  {n:<10}{ok:<6}{err:<8}{tk}")
    print("-" * (w + 36))
    print(f"{'ВСЕГО'.ljust(w)}  {sum(r[1] for r in summary):<10}{sum(r[2] for r in summary):<6}"
          f"{sum(r[3] for r in summary):<8}{sum(r[4] for r in summary)}")

# Запуск программы

if __name__ == "__main__":
    run_all()


Файл: 1_correct_easy.txt


[ OK    ] #1   найти все машины из таблицы 1 если марка toyota


[ OK    ] #2   найти все машины из таблицы 1 если марка равна toyota


[ OK    ] #3   показать машины из таблицы 2 если цвет красный


[ OK    ] #4   выбрать один автомобиль из таблицы cars если год 2020


[ OK    ] #5   удалить авто из таблицы 5 если пробег больше 100000


... (13 outputs left out)

[ OK    ] #3   найти один автомобиль из таблицы 3 если марка bmw и модель x5 и год 2019 и цвет черный или марка audi и мощность больше 200


[ OK    ] #4   показать все машины из таблицы 1 если владелец "Иван Петров" и цена не больше 1500000


[ OK    ] #5   найти все машины из таблицы 1 если год не равен 2010 и пробег не меньше 1000 или привод "полный" и мощность больше 300


[ OK    ] #6   выбрать все машины из таблицы 12 если марка honda и модель civic и год больше 2015 и пробег меньше 80000 и цена меньше 1200000 и цвет серый

СТАТИСТИКА ПО ФАЙЛУ
Число предложений (запросов): 6
успешно разобрано: 6
ошибок парсинга:  0
Число токенов (без EOF): 127
Среднее токенов на запрос: 21.2
Максимум / минимум: 27 / 14
Токены по типам: WORD=108, NUMBER=17, STRING=2

По разделам:
  Без раздела  OK: 6   ОШИБКА: 0

СВОДНАЯ ТАБЛИЦА ПО ЗАПРОСАМ
№   Статус   Токенов  WORD  NUMBER  STRING  Ошибка на токене
------------------------------------------------------------
1   OK       20       17    3       0       -
2   OK       18       17    1       0       -
3   OK       25       22    3       0       -
4   OK       14       11    2       1       -
5   OK       23       18    4       1       -
6   OK       27       23    4       0       -

ТАБЛИЦА ТОКЕНОВ
Запрос  Поз.   Тип      Значение
--------------------------------------------------
#1      0      WORD     найти
#1      6 

In [61]:
# Интерактивный режим: ввод запросов с клавиатуры

def interactive_mode():

    print("=" * 70)
    print("ИНТЕРАКТИВНЫЙ РЕЖИМ")
    print("Введите запрос или 'выход' для завершения")
    print("=" * 70)

    queries = []  # список всех запросов
    results = []  # список результатов (True/False)
    query_num = 0

    while True:
        try:
            query = input(f"\nЗапрос #{len(queries) + 1}: ").strip()
        except (KeyboardInterrupt, EOFError):
            # Обработка Ctrl+C и Ctrl+D
            print("\n\nЗавершение работы...")
            break

        # Проверка на выход
        if not query or query.lower() in {"выход", "exit", "quit"}:
            print("\nЗавершение работы...")
            break

        query_num += 1
        queries.append(query)

        # Разбор запроса
        print("-" * 70)
        tokens, ok, err = check_query(query)

        if ok:
            print(f"[ OK ] Запрос успешно разобран")
            results.append(True)
        else:
            print(f"[ ОШИБКА ] {error_text(err)}")
            results.append(False)

    # Статистика по сессии
    if queries:
        total = len(queries)
        ok_count = sum(results)
        err_count = total - ok_count

        print("\n" + "=" * 70)
        print("СТАТИСТИКА СЕССИИ")
        print("=" * 70)
        print(f"Всего запросов: {total}")
        print(f"Успешно разобрано: {ok_count}")
        print(f"Ошибок: {err_count}")
        if total > 0:
            print(f"Точность: {ok_count / total * 100:.1f}%")

        # Таблица результатов
        print("\n" + "=" * 70)
        print("СВОДНАЯ ТАБЛИЦА")
        print("=" * 70)
        print(f"{'№':<4}{'Статус':<9}Запрос")
        print("-" * 70)
        for i, (query, ok) in enumerate(zip(queries, results), 1):
            status = "OK" if ok else "ОШИБКА"
            print(f"{i:<4}{status:<9}{query}")


# Запуск интерактивного режима
if __name__ == "__main__":
    interactive_mode()

# найти все машины из таблицы 1 если марка toyota
# показать машины из таблицы 2 если пробег больше 100000
# удалить авто из таблицы 3 если цвет не красный
# найти все машины из таблицы 1 если марка bmw и модель x5 или марка audi и мощность больше 200
# выбрать один автомобиль из таблицы "основной автопарк" если владелец "Иван Петров" и цена больше 1500000.50

ИНТЕРАКТИВНЫЙ РЕЖИМ
Введите запрос или 'выход' для завершения
----------------------------------------------------------------------


[ OK ] Запрос успешно разобран

Завершение работы...

СТАТИСТИКА СЕССИИ
Всего запросов: 1
Успешно разобрано: 1
Ошибок: 0
Точность: 100.0%

СВОДНАЯ ТАБЛИЦА
№   Статус   Запрос
----------------------------------------------------------------------
1   OK       найти все машины из таблицы 1 если марка bmw и модель x5 или марка audi и мощность больше 200
